<a href="https://colab.research.google.com/github/VicKingo98/CopyBlock-Market/blob/master/S3P1_proyecto1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![image info](https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2023/main/images/banner_1.png)

# Proyecto 1 - Predicción de popularidad en canción

En este proyecto podrán poner en práctica sus conocimientos sobre modelos predictivos basados en árboles y ensambles, y sobre la disponibilización de modelos. Para su desarrollo tengan en cuenta las instrucciones dadas en la "Guía del proyecto 1: Predicción de popularidad en canción".

**Entrega**: La entrega del proyecto deberán realizarla durante la semana 4. Sin embargo, es importante que avancen en la semana 3 en el modelado del problema y en parte del informe, tal y como se les indicó en la guía.

Para hacer la entrega, deberán adjuntar el informe autocontenido en PDF a la actividad de entrega del proyecto que encontrarán en la semana 4, y subir el archivo de predicciones a la competencia de Kaggle cuyo link estará disponible en la sección del Coursera del proyecto.

## Datos para la predicción de popularidad en cancion

En este proyecto se usará el conjunto de datos de datos de popularidad en canciones, donde cada observación representa una canción y se tienen variables como: duración de la canción, acusticidad y tempo, entre otras. El objetivo es predecir qué tan popular es la canción. Para más detalles puede visitar el siguiente enlace: [datos](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset).

## Ejemplo predicción conjunto de test para envío a Kaggle

En esta sección encontrarán el formato en el que deben guardar los resultados de la predicción para que puedan subirlos a la competencia en Kaggle.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Importación librerías
import pandas as pd
import numpy as np

In [3]:
# Carga de datos de archivo .csv
dataTraining = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTrain_Spotify.csv')
dataTesting = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTest_Spotify.csv')

In [4]:
# Visualización datos de entrenamiento
dataTraining = dataTraining.drop(columns=['Unnamed: 0'])
dataTraining.head()

,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,popularity
0,7hUhmkALyQ8SX9mJs5XI3D,Love and Rockets,Love and Rockets,Motorcycle,211533,False,0.305,0.8490,9,-10.795,1,0.0549,0.000058,0.056700,0.4640,0.3200,141.793,4,goth,22
1,5x59U89ZnjZXuNAAlc8X1u,Filippa Giordano,Filippa Giordano,"Addio del passato - From ""La traviata""",196000,False,0.287,0.1900,7,-12.030,0,0.0370,0.930000,0.000356,0.0834,0.1330,83.685,4,opera,22
2,70Vng5jLzoJLmeLu3ayBQq,Susumu Yokota,Symbol,Purple Rose Minuet,216506,False,0.583,0.5090,1,-9.661,1,0.0362,0.777000,0.202000,0.1150,0.5440,90.459,3,idm,37
3,1cRfzLJapgtwJ61xszs37b,Franz Liszt;YUNDI,Relajación y siestas,"Liebeslied (Widmung), S. 566",218346,False,0.163,0.0368,8,-23.149,1,0.0472,0.991000,0.899000,0.1070,0.0387,69.442,3,classical,0
4,47d5lYjbiMy0EdMRV8lRou,Scooter,Scooter Forever,The Darkside,173160,False,0.647,0.9210,2,-7.294,1,0.1850,0.000939,0.371000,0.1310,0.1710,137.981,4,techno,27


In [5]:
# Visualización datos de test
dataTesting = dataTesting.drop(columns=['Unnamed: 0'])
dataTesting.head()

,track_id,artists,album_name,track_name,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,6KwkVtXm8OUp2XffN5k7lY,Hillsong Worship,No Other Name,No Other Name,440247,False,0.369,0.598,7,-6.984,1,0.0304,0.00511,0.000000,0.176,0.0466,148.014,4,world-music
1,2dp5I5MJ8bQQHDoFaNRFtX,Internal Rot,Grieving Birth,Failed Organum,93933,False,0.171,0.997,7,-3.586,1,0.1180,0.00521,0.801000,0.420,0.0294,122.223,4,grindcore
2,5avw06usmFkFrPjX8NxC40,Zhoobin Askarieh;Ali Sasha,Noise A Noise 20.4-1,"Save the Trees, Pt. 1",213578,False,0.173,0.803,9,-10.071,0,0.1440,0.61300,0.001910,0.195,0.0887,75.564,3,iranian
3,75hT0hvlESnDJstem0JgyR,Bryan Adams,All I Want For Christmas Is You,Merry Christmas,151387,False,0.683,0.511,6,-5.598,1,0.0279,0.40600,0.000197,0.111,0.5980,109.991,3,rock
4,4bY2oZGA5Br3pTE1Jd1IfY,Nogizaka46,バレッタ TypeD,月の大きさ,236293,False,0.555,0.941,9,-3.294,0,0.0481,0.48400,0.000000,0.266,0.8130,92.487,4,j-idol


In [6]:
!pip install xgboost

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score

X = dataTraining.drop(columns=['popularity'])
y = dataTraining['popularity']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# FEATURE ENGINEERING

def feature_engineering(df):
    df = df.copy()

    # duración
    df['duration_min'] = df['duration_ms'] / 60000

    # número de artistas
    df['n_artists'] = df['artists'].apply(
        lambda x: len(x.split(';')) if pd.notnull(x) else 0
    )

    return df


X_train = feature_engineering(X_train)
X_val = feature_engineering(X_val)
dataTesting = feature_engineering(dataTesting)


# Reducción de Cardinalidad
top_genres = X_train['track_genre'].value_counts().head(15).index

def reduce_genres(df):
    df['track_genre'] = df['track_genre'].apply(
        lambda x: x if x in top_genres else 'other'
    )
    return df

X_train = reduce_genres(X_train)
X_val = reduce_genres(X_val)
dataTesting = reduce_genres(dataTesting)


# Genre Popularity
def kfold_target_encoding(X, y, column, n_splits=5):
    X = X.copy()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    global_mean = y.mean()
    X[column + '_te'] = 0

    for train_idx, val_idx in kf.split(X):
        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]

        means = pd.concat([X_train_fold, y_train_fold], axis=1) \
                  .groupby(column)['popularity'].mean()

        X.iloc[val_idx, X.columns.get_loc(column + '_te')] = \
            X_val_fold[column].map(means)

    X[column + '_te'] = X[column + '_te'].fillna(global_mean)

    return X

X_train = kfold_target_encoding(X_train, y_train, 'track_genre')

genre_means = X_train.groupby('track_genre')['track_genre_te'].mean()

X_val['track_genre_te'] = X_val['track_genre'].map(genre_means)
X_val['track_genre_te'] = X_val['track_genre_te'].fillna(y_train.mean())

dataTesting['track_genre_te'] = dataTesting['track_genre'].map(genre_means)
dataTesting['track_genre_te'] = dataTesting['track_genre_te'].fillna(y_train.mean())

# eliminar columna original categórica
X_train = X_train.drop(columns=['track_genre'])
X_val = X_val.drop(columns=['track_genre'])
dataTesting = dataTesting.drop(columns=['track_genre'])

# Artist Popularity
artist_stats = pd.concat([X_train[['artists']], y_train], axis=1)

artist_mean = artist_stats.groupby('artists')['popularity'].mean()
artist_count = artist_stats.groupby('artists')['popularity'].count()

global_mean = y_train.mean()

artist_pop = (artist_mean * artist_count + global_mean * 10) / (artist_count + 10)

X_train['artist_popularity'] = X_train['artists'].map(artist_pop)
X_train['artist_popularity'] = X_train['artist_popularity'].fillna(global_mean)
X_val['artist_popularity'] = X_val['artists'].map(artist_pop)
dataTesting['artist_popularity'] = dataTesting['artists'].map(artist_pop)

X_val['artist_popularity'] = X_val['artist_popularity'].fillna(global_mean)
dataTesting['artist_popularity'] = dataTesting['artist_popularity'].fillna(global_mean)

def get_max_artist_pop(artists_str):
    if pd.isnull(artists_str):
        return global_mean
    return max([artist_pop.get(a, global_mean) for a in artists_str.split(';')])

X_train['artist_max_pop'] = X_train['artists'].apply(get_max_artist_pop)
X_val['artist_max_pop'] = X_val['artists'].apply(get_max_artist_pop)
dataTesting['artist_max_pop'] = dataTesting['artists'].apply(get_max_artist_pop)

X_train['is_collab'] = (X_train['n_artists'] > 1).astype(int)
X_val['is_collab'] = (X_val['n_artists'] > 1).astype(int)
dataTesting['is_collab'] = (dataTesting['n_artists'] > 1).astype(int)

X_train['tempo_bin'] = pd.cut(X_train['tempo'], bins=5, labels=False)
X_train['tempo_bin'] = X_train['tempo_bin'].astype(float).fillna(0)
X_val['tempo_bin'] = pd.cut(X_val['tempo'], bins=5, labels=False)
X_val['tempo_bin'] = X_val['tempo_bin'].astype(float).fillna(0)
dataTesting['tempo_bin'] = pd.cut(dataTesting['tempo'], bins=5, labels=False)
dataTesting['tempo_bin'] = dataTesting['tempo_bin'].astype(float).fillna(0)

def scale_loudness(df):
    df['loudness_scaled'] = (df['loudness'] + 60) / 60
    return df

X_train = scale_loudness(X_train)
X_val = scale_loudness(X_val)
dataTesting = scale_loudness(dataTesting)

# Frecuencia de género
genre_freq = X_full['track_genre'].value_counts()
X_train['genre_freq'] = X_train['track_genre'].map(genre_freq)
X_val['genre_freq'] = X_val['track_genre'].map(genre_freq)
dataTesting['genre_freq'] = dataTesting['track_genre'].map(genre_freq)

# Frecuencia de artista
artist_freq = X_full['artists'].value_counts()
X_train['artist_freq'] = X_train['artists'].map(artist_freq)
X_val['artist_freq'] = X_val['artists'].map(artist_freq)
dataTesting['artist_freq'] = dataTesting['artists'].map(artist_freq)



# Interacciones
def add_interactions(df):
    df['energy_valence'] = df['energy'] * df['valence']
    df['dance_energy'] = df['danceability'] * df['energy']
    df['energy_loudness'] = df['energy'] * df['loudness_scaled']
    df['tempo_energy'] = df['tempo'] * df['energy']
    df['loudness_valence'] = df['loudness_scaled'] * df['valence']
    df['tempo_dance'] = df['tempo'] * df['danceability']
    df['duration_bin'] = pd.cut(df['duration_min'], bins=[0,2,3,4,10], labels=False)
    df['duration_bin'] = df['duration_bin'].astype(float).fillna(0)
    return df

X_train = add_interactions(X_train)
X_val = add_interactions(X_val)
dataTesting = add_interactions(dataTesting)

In [9]:
cols_to_drop = ['track_id', 'artists', 'album_name', 'track_name']

X_train = X_train.drop(columns=cols_to_drop)
X_val = X_val.drop(columns=cols_to_drop)
dataTesting = dataTesting.drop(columns=cols_to_drop)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
dataTesting = dataTesting.reindex(columns=X_train.columns, fill_value=0)

In [10]:
# Inicialización del modelo para calibrar
xgb = XGBRegressor(random_state=42, n_jobs=-1)

In [11]:
# Definición de búsqueda para los hiperparámetros
param_dist = {
    'n_estimators': [500, 600, 700],
    'max_depth': [6, 7, 8],
    'learning_rate': [0.01, 0.03],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.5, 1, 2],
    'reg_lambda': [1, 2, 3, 5]
}


# Calibración de hiperparámetros y selección del mejor modelo
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=10,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42
)

random_search.fit(X_train, y_train)

best_model = random_search.best_estimator_

scores = cross_val_score(
    best_model,
    X_train,
    y_train,
    scoring='neg_root_mean_squared_error',
    cv=5
)

# Evaluación en conjunto de validation
y_pred_xgb_val = best_model.predict(X_val)

rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb_val))

print("Best params:", random_search.best_params_)
print("RMSE CV:", -scores.mean())
print("STD CV:", scores.std())
print("RMSE XGB:",rmse_xgb)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params: {'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 2, 'n_estimators': 700, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 0.6}
RMSE CV: 11.64005012512207
STD CV: 0.14763423392909675
RMSE XGB: 14.634262933654298


In [12]:
# Implementación Random Forest
X_full = pd.concat([X_train, X_val])
y_full = pd.concat([y_train, y_val])

rf = RandomForestRegressor(
    n_estimators=250,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

kf = KFold(n_splits=3, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(X_full))
oof_rf = np.zeros(len(X_full))

for train_idx, val_idx in kf.split(X_full):

    X_tr, X_val_fold = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_val_fold = y_full.iloc[train_idx], y_full.iloc[val_idx]

    # XGB
    best_model.fit(X_tr, y_tr)
    oof_xgb[val_idx] = best_model.predict(X_val_fold)

    # RF
    rf.fit(X_tr, y_tr)
    oof_rf[val_idx] = rf.predict(X_val_fold)

In [13]:
# Meta Model
X_meta = np.column_stack((oof_xgb, oof_rf))

meta_model = Ridge(alpha=1.0)
meta_model.fit(X_meta, y_full)

Ridge()

In [14]:
y_pred_oof = meta_model.predict(X_meta)
rmse_oof = np.sqrt(mean_squared_error(y_full, y_pred_oof))
print("RMSE OOF:", rmse_oof)

RMSE OOF: 12.198334495107346


In [16]:
# Re entrenamiento con todo el dataset
best_model.fit(X_full, y_full)
rf.fit(X_full, y_full)

RandomForestRegressor(max_depth=10, n_estimators=500, n_jobs=-1,
                      random_state=42)

In [17]:
# limpieza final
dataTesting = dataTesting.drop(columns=['popularity'], errors='ignore')

# alinear columnas exactas
dataTesting = dataTesting.reindex(columns=X_full.columns, fill_value=0)
dataTesting = dataTesting.fillna(0)

# predicción
y_pred_xgb_test = best_model.predict(dataTesting)
y_pred_rf_test = rf.predict(dataTesting)

X_meta_test = np.column_stack((y_pred_xgb_test, y_pred_rf_test))

y_pred_test = meta_model.predict(X_meta_test)

# clip
y_pred_test = np.clip(y_pred_test, 0, 100)

In [18]:
# generación de submission
submission = pd.DataFrame({
    "Popularity": y_pred_test
}, index=dataTesting.index)

submission.to_csv("submissionvf.csv", index_label="ID")